In [1]:
%reset -f

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

Check for cuda

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

Using device: cuda


Load data

In [4]:
path = Path.cwd().parent.parent / "TabulatedData" / "values_v6_32bit.parquet"

dataset = pd.read_parquet(path)

X = dataset.drop("name", axis=1).values
y = dataset["name"].values

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
y_one_hot = encoder.fit_transform(pd.DataFrame(y))
y = np.argmax(y_one_hot, axis=1)

# Get class names from encoder
class_names = encoder.categories_[0]

print("Data Loaded")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

del dataset, X, y

# -----------------------------
# Feature Scaling (recommended for MLP)
# -----------------------------
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Data Loaded


To PyTorch Tensors

In [5]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

Define Model

In [6]:
input_size = X_train.shape[1]
num_classes = len(class_names)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -----------------------------
# Loss and Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training Loop

In [7]:
epochs = 1000

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss:.4f}")


Epoch [100/1000], Loss: 4.3666
Epoch [200/1000], Loss: 53.8054
Epoch [300/1000], Loss: 3.3131
Epoch [400/1000], Loss: 3.2365
Epoch [500/1000], Loss: 3.1278
Epoch [600/1000], Loss: 4.9393
Epoch [700/1000], Loss: 3.0328
Epoch [800/1000], Loss: 6.5371
Epoch [900/1000], Loss: 2.9192
Epoch [1000/1000], Loss: 2.9192


Evaluation Test

In [8]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    preds = torch.argmax(outputs, dim=1)

acc = accuracy_score(y_test.cpu().numpy(), preds.cpu().numpy())

print(f"\nTest Accuracy: {acc:.4f}")
print(classification_report(y_test.cpu().numpy(), preds.cpu().numpy(), target_names=class_names, zero_division=0))


Test Accuracy: 0.1932
                    precision    recall  f1-score   support

      Aditya Kundu       0.40      0.43      0.42        23
       Akash Gupta       0.24      0.21      0.22        56
          Ankur De       0.41      0.41      0.41       173
           Ashtavi       0.36      0.40      0.38        25
          Avyuktha       0.03      0.03      0.03       112
        Harshith H       0.33      0.38      0.36        34
     Jiya Sachdeva       0.28      0.26      0.27        78
     Karthikeya SK       0.18      0.12      0.14        26
    Nandini Sharma       0.27      0.27      0.27        51
           Navnita       0.11      0.16      0.13       105
          Papia De       0.53      0.44      0.48        59
        Pavithra S       0.49      0.40      0.44       100
            Piyali       0.33      0.23      0.27        26
Prajwal Mundiganal       0.71      0.57      0.64        61
           Ruthvik       0.38      0.32      0.35       102
            Tris

Evaluation Train

In [9]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)
    preds = torch.argmax(outputs, dim=1)

acc = accuracy_score(y_train.cpu().numpy(), preds.cpu().numpy())

print(f"\nTrain Accuracy: {acc:.4f}")


Test Accuracy: 0.9873
